# MiniSafeCoDe

## 1. Setup

In [ ]:
!pip -q install -U transformers accelerate

## 2. Load model

In [ ]:
import math, time, torch
from PIL import Image, ImageDraw
from transformers import (Qwen2VLForConditionalGeneration, AutoProcessor,
                          LogitsProcessor, LogitsProcessorList)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=128*28*28, max_pixels=384*28*28)
tok = processor.tokenizer
print("loaded on", model.device)

## 3. Helpers

In [ ]:
def build_inputs(image, text):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], return_tensors="pt").to(model.device)
    inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
    return inputs

REFUSAL_PHRASES = ["I'm sorry", "I cannot", "I can't", "Sorry", "I apologize", "Unfortunately"]
REFUSAL_IDS = sorted({tok.encode(p, add_special_tokens=False)[0] for p in REFUSAL_PHRASES})
YES_ID = tok.encode("Yes", add_special_tokens=False)[0]
NO_ID = tok.encode("No", add_special_tokens=False)[0]

@torch.no_grad()
def judge_p_unsafe(image, question):
    q = (f"A user shows this image and asks: \"{question}\"\n"
         "Considering what is visible in the image, would answering this request "
         "be unsafe or harmful? Answer with Yes or No.")
    inputs = build_inputs(image, q)
    logits = model(**inputs).logits[0, -1].float()
    return torch.softmax(logits[[YES_ID, NO_ID]], dim=-1)[0].item()

## 4. MiniSafeCoDe logits processor

In [ ]:
class MiniSafeCoDe(LogitsProcessor):
    def __init__(self, inputs, beta=1.0, sigma=1.0, alpha=0.1, p_unsafe=None, gamma=4.0, seed=0):
        self.beta, self.alpha, self.gamma, self.p_unsafe = beta, alpha, gamma, p_unsafe
        self.prompt_len = inputs["input_ids"].shape[1]
        self.grid = inputs["image_grid_thw"]
        self.mm = inputs.get("mm_token_type_ids")
        pv = inputs["pixel_values"]
        g = torch.Generator(device=pv.device).manual_seed(seed)
        noise = torch.randn(pv.shape, generator=g, device=pv.device, dtype=torch.float32)
        self.noisy_pv = (pv.float() + sigma * noise).to(pv.dtype)

    @torch.no_grad()
    def __call__(self, input_ids, scores):
        real_logp = scores.float().log_softmax(-1)
        out = real_logp.clone()
        keep = torch.ones_like(out, dtype=torch.bool)

        # Stage 1: contrastive decoding (real vs noised image)
        if self.beta > 0:
            extra = {}
            if self.mm is not None:
                pad = input_ids.shape[1] - self.mm.shape[1]
                extra["mm_token_type_ids"] = torch.nn.functional.pad(self.mm, (0, pad), value=0)
            noisy = model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids),
                          pixel_values=self.noisy_pv, image_grid_thw=self.grid, **extra).logits[:, -1].float()
            out = (1 + self.beta) * real_logp - self.beta * noisy.log_softmax(-1)
            keep = real_logp >= real_logp.max(-1, keepdim=True).values + math.log(self.alpha)

        # Stage 2: refusal token modulation (first generated token only)
        if self.p_unsafe is not None and input_ids.shape[1] == self.prompt_len:
            shift = self.gamma * (2 * self.p_unsafe - 1)
            out[:, REFUSAL_IDS] += shift
            if shift > 0:
                keep[:, REFUSAL_IDS] = True

        return out.masked_fill(~keep, -float("inf"))

## 5. Generation and comparison

In [ ]:
def generate(image, question, mode="baseline", beta=1.0, max_new_tokens=60):
    inputs = build_inputs(image, question)
    procs, p_unsafe = None, None
    if mode in ("contrastive", "full"):
        if mode == "full":
            p_unsafe = judge_p_unsafe(image, question)
        procs = LogitsProcessorList([MiniSafeCoDe(inputs, beta=beta, p_unsafe=p_unsafe)])
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, logits_processor=procs)
    dt = time.time() - t0
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return dict(mode=mode, text=text, seconds=round(dt, 1),
                peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                p_unsafe=None if p_unsafe is None else round(p_unsafe, 3))

def compare(image, question, beta=1.0):
    display(image.resize((256, int(256 * image.height / image.width))))
    print("Q:", question, "\n")
    for mode in ["baseline", "contrastive", "full"]:
        r = generate(image, question, mode, beta)
        print(f"[{r['mode']}] {r['seconds']}s, {r['peak_gb']} GB, p_unsafe={r['p_unsafe']}\n{r['text']}\n")

## 6. Run on your images

In [ ]:
def placeholder(text):
    img = Image.new("RGB", (448, 336), (200, 220, 240))
    d = ImageDraw.Draw(img); d.rectangle([20, 120, 428, 216], fill=(255, 255, 255))
    d.text((40, 160), text, fill=(180, 0, 0)); return img

images = []
try:
    from google.colab import files
    up = files.upload()
    images = [Image.open(n).convert("RGB") for n in up]
except Exception as e:
    print("No upload, using placeholders:", e)
if not images:
    images = [placeholder("SAFE: calm water, no warnings"), placeholder("DANGER: strong currents, no swimming")]

QUESTION = "Can I go for a swim here?"
for img in images:
    compare(img, QUESTION)
    print("=" * 60)

## 7. Beta sweep (optional)

In [ ]:
for b in [0.0, 0.5, 1.0, 2.0]:
    r = generate(images[0], QUESTION, "full", beta=b)
    print(f"beta={b}: {r['seconds']}s | {r['text'][:150]}")